### ***Attention Is All You Need***

In [33]:
# all the imports
import torch
import torch.nn as nn
import math

In [34]:
## input embedding class

# __init__
#input : d_model : dimension of the model ,vocab_size : size of the vocabulary

#forward
#input : x : input tokens of shape (batch_size, seq_len)
#output : embedding of the input tokens
class InputEmbeddings(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.embedding(x) * (self.d_model ** 0.5)


In [35]:
## creating positional encoding class

# __init__
#input: d_model: dimension of the model, seq_len: maximum sequence length, dropout: dropout rate
# output: positional encoding matrix of shape (seq_len, d_model)

# forward
# input: x: input tensor of shape (batch_size, seq_len, d_model) : (embedding of the input sequence)
# output: tensor of shape (batch_size, seq_len, d_model) with positional encoding added

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, seq_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        # creating positional encoding matrix (seq_len, d_model)
        pe = torch.zeros(seq_len, d_model)

        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)  ## => [0, 1, 2, ..., seq_len-1] shape (seq_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))  ## shape (d_model/2,)

        pe[:, 0::2] = torch.sin(position * div_term)  ## even indices
        pe[:, 1::2] = torch.cos(position * div_term)  ## odd indices

        pe = pe.unsqueeze(0)  # Add a batch dimension : for shape (1, seq_len, d_model)
        self.register_buffer('pe', pe) # registering the positional encoding matrix as a buffer so that it is not considered a model parameter and will not be updated during training

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1), :] #positional encoding is not updated during training as this is registered as a buffer and not a parameter of the model
        return self.dropout(x)

In [36]:
## layer normalization class

#__init__
#input: d_model: dimension of the model, eps: small value to avoid division by zero

#forward
#input: x: input tensor of shape (batch_size, seq_len, d_model)
#output: tensor of shape (batch_size, seq_len, d_model) with layer normalization

class LayerNormalization(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.d_model = d_model
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model)) # making it learnable parameter
        self.beta = nn.Parameter(torch.zeros(d_model)) # making it learnable parameter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        # we are calculating the std directly , not the var => this is the small from the actual paper
        
        x_hat = (x - mean) / (std + self.eps) # normalizing the input tensor
        return self.gamma * x_hat + self.beta # scaling and shifting the normalized tensor

In [37]:
## Feed Forward Network class

#architecture : x(batch_size,seq_len,d_model)
#                       ↓
#              Linear(d_model,d_ff) (d_ff=2048 neurons)
#                      ↓
#                    ReLU
#                      ↓
#                  Dropout
#                     ↓
#              Linear(d_ff,d_model) (d_model=512 neurons) 
#                     ↓
#           output(batch_size,seq_len,d_model)  
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

In [38]:
# multi head attention class

# __init__
# input: d_model: dimension of the model, num_heads: number of attention heads, dropout: dropout rate
# creating the multi head attention layer with linear projections for query, key, value and output

# attention
# input: query, key, value: input tensors of shape (batch_size, num_heads, seq_len, head_dim)
#        mask: optional mask tensor of shape (batch_size, seq_len, seq_len)
#        dropout: dropout layer for attention weights
# output : the attention score of shape (batch_size, num_heads, seq_len, head_dim) and 
#          the attention weights ( softmax ( if mask then mask(sclaled dot product of query and key) 
#                                           else softmax(scaled dot product of query and key)  )


# forward
# input: query, key, value: input tensors of shape (batch_size, seq_len, d_model)
#        mask: optional mask tensor of shape (batch_size, seq_len, seq_len)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: torch.Tensor, dropout: nn.Dropout) -> torch.Tensor:
        # query, key, value: (batch_size, num_heads, seq_len, head_dim)
        head_dim = query.size(-1)

        # Scaled dot-product attention
        #  query , key : (batch_size, num_heads, seq_len_q, head_dim) , (batch_size, num_heads, seq_len_k, head_dim)
        #  key.transpose(-2, -1) : (batch_size, num_heads, head_dim, seq_len_k)
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(head_dim)  # scores : (batch_size, num_heads, seq_len_q, seq_len_k)

        # for masked multihead attention
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)  # (batch_size, num_heads, seq_len_q, seq_len_k)
        if dropout is not None:
            attention_weights = dropout(attention_weights)

        output = torch.matmul(attention_weights, value)  # (batch_size, num_heads, seq_len_q, head_dim)
        return output , attention_weights

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # query, key, value: (batch_size, seq_len, d_model)
        batch_size = query.size(0)
        seq_len_q = query.size(1)
        seq_len_k = key.size(1)
        seq_len_v = value.size(1)

        # Linear projections
        Q = self.w_q(query)  # (batch_size, seq_len, d_model)
        K = self.w_k(key)      # (batch_size, seq_len, d_model)
        V = self.w_v(value)  # (batch_size, seq_len, d_model)

        # Split into multiple heads
        # from each head , we want to get the sequences => so, we have transposed the views
        # after transposing , now we can have like this : from every head => we have all the words in the sequence of the head_dim dimensions 
        Q = Q.view(batch_size, seq_len_q, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len_q, head_dim)
        K = K.view(batch_size, seq_len_k, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len_k, head_dim)
        V = V.view(batch_size, seq_len_v, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len_v, head_dim)

        output, attention_weights = self.attention(Q, K, V, mask, self.dropout)
        
        # output: (batch_size, num_heads, seq_len, head_dim)
        # for geting the concatenated output of all the heads, first we need to transpose the output to (batch_size, seq_len, num_heads, head_dim)
        # before passing through the output linear layer, we need to reshape it back to (batch_size, seq_len, d_model)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len_q, self.d_model)  # (batch_size, seq_len_q, d_model)
        output = self.w_o(output)  # (batch_size, seq_len_q, d_model)

        return output

In [39]:
# residual connection class

# the flow will be like this : input -> sublayer -> dropout -> add input -> layer normalization -> output ( as in the original paper)
class ResidualConnection(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.layer_norm = LayerNormalization(d_model)
        self.dropout = nn.Dropout(dropout)

    # here sublayer is a function that takes input x and returns output of the sublayer 
    # the function can be => multihead attention or feed forward network
    def forward(self, x, sublayer): 
        sublayer_output = sublayer(x)
        return self.layer_norm(x + self.dropout(sublayer_output))

In [40]:
# encoder block class
# for each encoder block:
#                       input ( positional encoding + input embedding )
#                                   ↓
#                      residual connection_1 (sublayer function =  multi head attention )
#                                   ↓
#                      residual connection_2 (sublayer function =  feed forward network )
#                                   ↓
#                      output ( to the next encoder block or decoder )

class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.multi_head_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForwardNetwork(d_model, d_ff, dropout)
        self.residual_connection1 = ResidualConnection(d_model, dropout)
        self.residual_connection2 = ResidualConnection(d_model, dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # x: (batch_size, seq_len, d_model)
        # mask: (batch_size, seq_len, seq_len)
        x = self.residual_connection1(x, lambda x: self.multi_head_attention(x, x, x, mask)) # lambda function is used to pass the sublayer function to the residual connection
        x = self.residual_connection2(x, self.feed_forward)
        return x

In [41]:
# encoder layer class

# input : num_layers : number of encoder blocks , d_model : dimension of the model , num_heads : number of attention heads , d_ff : dimension of the feed forward network , dropout : dropout rate
# output : tensor of shape (batch_size, seq_len, d_model) after passing through all the encoder blocks
class Encoder(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.layers = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.layer_norm = LayerNormalization(d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, mask) # each encoder block forwards the output to the next encoder block
        x = self.layer_norm(x)
        return x

In [42]:
# decoder block class
# for each decoder block:
#                       input ( positional encoding + input embedding )
#                                   ↓
#                      residual connection_1 (sublayer function =  masked multi head attention )
#                                   ↓
#                      residual connection_2 (sublayer function = cross multi head attention )
#                                   ↓
#                      residual connection_3 (sublayer function =  feed forward network )
#                                   ↓
#                        output ( to the next decoder block or final linear layer )

class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.masked_multi_head_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_multi_head_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForwardNetwork(d_model, d_ff, dropout)
        self.residual_connection1 = ResidualConnection(d_model, dropout)
        self.residual_connection2 = ResidualConnection(d_model, dropout)
        self.residual_connection3 = ResidualConnection(d_model, dropout)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor, src_mask: torch.Tensor = None, tgt_mask: torch.Tensor = None) -> torch.Tensor:
        # x: (batch_size, seq_len_tgt, d_model)
        # encoder_output: (batch_size, seq_len_src, d_model)
        # src_mask: (batch_size, seq_len_src, seq_len_src) => Hide padding/invalid source tokens 
        # tgt_mask: (batch_size, seq_len_tgt, seq_len_tgt) => hide future tokens and padding/invalid target tokens
        x = self.residual_connection1(x, lambda x: self.masked_multi_head_attention(x, x, x, tgt_mask)) # lambda function is used to pass the sublayer function to the residual connection
        x = self.residual_connection2(x, lambda x: self.cross_multi_head_attention(x, encoder_output, encoder_output, src_mask)) # lambda function is used to pass the sublayer function to the residual connection
        x = self.residual_connection3(x, self.feed_forward)
        return x

In [43]:
# decoder layer class

class Decoder(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.layers = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.layer_norm = LayerNormalization(d_model)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor, src_mask: torch.Tensor = None, tgt_mask: torch.Tensor = None) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask) # each decoder block forwards the output to the next decoder block
        x = self.layer_norm(x)
        return x

In [44]:
# projecting the output of the decoder to the vocabulary size for generating the final output probabilities

#input : d_model : dimension of the model , vocab_size : size of the vocabulary
#output : tensor of shape (batch_size, seq_len, vocab_size) after passing through the linear layer and log softmax activation function
# forward : input : x : input : tensor of shape (batch_size, seq_len, d_model) ,
#                       output : tensor of shape (batch_size, seq_len, vocab_size) 
class ProjectionToVocab(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.linear = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.log_softmax(self.linear(x), dim=-1)  # output shape: (batch_size, seq_len, vocab_size)


In [45]:
class Transformer(nn.Module):

    def __init__(
        self,
        num_encoder_layers: int,
        num_decoder_layers: int,
        d_model: int,
        num_heads: int,
        d_ff: int,
        input_vocab_size: int,
        output_vocab_size: int,
        max_seq_length: int,
        dropout: float = 0.1
    ):
        super().__init__()

        # Separate embedding layers
        self.input_embedding = InputEmbeddings(
            d_model,
            input_vocab_size
        )

        self.output_embedding = InputEmbeddings(
            d_model,
            output_vocab_size
        )

        self.positional_encoding = PositionalEncoding(
            d_model,
            max_seq_length
        )

        self.encoder = Encoder(
            num_encoder_layers,
            d_model,
            num_heads,
            d_ff,
            dropout
        )

        self.decoder = Decoder(
            num_decoder_layers,
            d_model,
            num_heads,
            d_ff,
            dropout
        )

        self.projection_to_vocab = ProjectionToVocab(
            d_model,
            output_vocab_size
        )

        self._init_weights()

    # check for the initialization of the weights of the model parameters using Xavier uniform initialization for parameters with more than one dimension.
    def _init_weights(self):

        for parameter in self.parameters():

            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)

    def encode(
        self,
        src: torch.Tensor,
        src_mask: torch.Tensor = None
    ) -> torch.Tensor:

        src_embedded = self.input_embedding(src)
        # (batch_size, seq_len_src, d_model)

        src_positional_encoded = self.positional_encoding(src_embedded)
        # (batch_size, seq_len_src, d_model)

        src_encoded = self.encoder(
            src_positional_encoded,
            src_mask
        )
        # (batch_size, seq_len_src, d_model)

        return src_encoded

    def decode(
        self,
        tgt: torch.Tensor,
        src_encoded: torch.Tensor,
        src_mask: torch.Tensor = None,
        tgt_mask: torch.Tensor = None
    ) -> torch.Tensor:

        tgt_embedded = self.output_embedding(tgt)
        # (batch_size, seq_len_tgt, d_model)

        tgt_positional_encoded = self.positional_encoding(tgt_embedded)
        # (batch_size, seq_len_tgt, d_model)

        tgt_decoded = self.decoder(
            tgt_positional_encoded,
            src_encoded,
            src_mask,
            tgt_mask
        )
        # (batch_size, seq_len_tgt, d_model)

        return tgt_decoded

    def forward(
        self,
        src: torch.Tensor,
        tgt: torch.Tensor,
        src_mask: torch.Tensor = None,
        tgt_mask: torch.Tensor = None
    ) -> torch.Tensor:

        # src: (batch_size, seq_len_src)
        # tgt: (batch_size, seq_len_tgt)

        # src_mask:
        # (batch_size, seq_len_src, seq_len_src)

        # tgt_mask:
        # (batch_size, seq_len_tgt, seq_len_tgt)

        src_encoded = self.encode(
            src,
            src_mask
        )

        tgt_decoded = self.decode(
            tgt,
            src_encoded,
            src_mask,
            tgt_mask
        )

        output = self.projection_to_vocab(tgt_decoded)
        # (batch_size, seq_len_tgt, output_vocab_size)

        return output

In [46]:
class BuildTransformer:

    def __init__(
        self,
        num_encoder_layers: int,
        num_decoder_layers: int,
        d_model: int,
        num_heads: int,
        d_ff: int,
        input_vocab_size: int,
        output_vocab_size: int,
        max_seq_length: int,
        dropout: float = 0.1
    ):

        self.transformer = Transformer(
            num_encoder_layers,
            num_decoder_layers,
            d_model,
            num_heads,
            d_ff,
            input_vocab_size,
            output_vocab_size,
            max_seq_length,
            dropout
        )

    def get_model(self):
        return self.transformer